# Tutorial 1: Basic Optimization

Learn the complete optimization workflow from project setup to result analysis through a realistic plasma etching calibration example.

**Estimated time:** 30-45 minutes

## What You'll Learn

- Create and configure a ViennaFit project
- Load experimental data as target domain
- Define a physics-based process sequence
- Set parameter bounds and choose distance metrics
- Run optimization and interpret results
- Validate and refine your calibration

## Scenario

You have SEM cross-section images of a plasma-etched silicon trench and want to calibrate your process model parameters to match the experimental profile.

**Process:** SF6/C4F8 plasma etching (Bosch process)

**Parameters to calibrate:**
- Ion source power (controls angular distribution)
- Neutral etchant rate
- Ion mean energy
- Neutral sticking probability
- Ion etch rate

**Goal:** Find parameters that make simulated profile match experimental cross-section

In [ ]:
import viennafit as fit
import viennaps as vps
import viennals as vls
import json
import os

# Set dimensions (2D simulation)
vps.setDimension(2)
vls.setDimension(2)

# Create project
project = fit.Project("etchCalibration", "./projects")
project.initialize()

print(f"Project created at: {project.projectPath}")

## Step 2: Set the Initial Domain for your project

Define the initial substrate geometry before etching. `vps.MakeTrench` with `makeMask=True`
creates a silicon substrate with a patterned mask (opening of `trenchWidth`) on top — no trench
cut into the substrate yet:

In [ ]:
# Grid resolution
gridDelta = 5.0  # nm

initialDomain = vps.Domain()
vps.MakeTrench(
    initialDomain,
    gridDelta=gridDelta,
    xExtent=600.0,       # 600nm wide
    yExtent=400.0,       # 400nm tall
    trenchWidth=200.0,   # mask opening width (x=-100 to x=100)
    trenchDepth=50.0,    # mask height (nm)
    makeMask=True,       # substrate + mask with opening; no trench yet
    material=vps.Material.Si
).apply()

project.setInitialDomain(initialDomain)
initialDomain.saveSurfaceMesh(
    os.path.join(project.projectPath, "domains/initialDomain/initial.vtp"),
    True  # Add material IDs
)

print("Initial domain created and saved")

## Step 3: Load Target Domain

In a real scenario, you'd load experimental data. For this tutorial, we'll create a target that
represents a typical etched trench. `vps.MakeTrench` with `makeMask=False` (default) creates a
silicon substrate with a trench already etched into it:

> **Real experimental data:** To load actual experimental data:
> ```python
> target = vls.Domain([...], boundary, gridDelta)
> reader = vls.VTKReader(target)
> reader.apply("experimental_profile.vtp")
> project.setTargetLevelSet(target)
> ```

In [ ]:
targetDomain = vps.Domain()
vps.MakeTrench(
    targetDomain,
    gridDelta=gridDelta,
    xExtent=600.0,       # 600nm wide
    yExtent=400.0,       # 400nm tall
    trenchWidth=200.0,   # trench width (x=-100 to x=100)
    trenchDepth=150.0,   # trench depth (150nm)
    material=vps.Material.Si
).apply()

project.setTargetLevelSet(targetDomain.getLevelSets()[-1])

targetMesh = vls.Mesh()
vls.ToSurfaceMesh(targetDomain.getLevelSets()[-1], targetMesh).apply()
writer = vls.VTKWriter(targetMesh)
writer.setFileName(
    os.path.join(project.projectPath, "domains/targetDomain/target.vtp")
)
writer.apply()

print("Target domain created and saved")

## Step 4: Define Process Sequence

Create a physics-based SF6/C4F8 plasma etching process:

In [ ]:
def etchingProcess(domain: vps.Domain, params: dict[str, float]) -> vls.Domain:
    """
    SF6/C4F8 plasma etching process.

    Parameters:
    - ionSourcePower: Source power controlling ion angular distribution
    - neutralRate: Neutral etchant etch rate contribution (nm/s)
    - ionEnergy: Ion mean energy (eV)
    - neutralStickP: Neutral sticking probability
    - ionRate: Ion etch rate contribution (nm/s)
    """
    # Set up model
    model = vps.MultiParticleProcess()

    # Add neutral etchant particles first (-> fluxes[0])
    sticking = {
        vps.Material.Si: params["neutralStickP"],
        vps.Material.SiO2: 0.01  # Low sticking on mask
    }
    model.addNeutralParticle(
        sticking=sticking,
        label="etchant"
    )

    # Add ion particles second (-> fluxes[1])
    model.addIonParticle(
        sourcePower=params["ionSourcePower"],
        meanEnergy=params["ionEnergy"],
        sigmaEnergy=10.0,  # Energy spread
        label="ion"
    )

    # Define etch rate function
    # fluxes[0] = neutral flux, fluxes[1] = ion flux
    # Rates are negative for etching (material removal)
    def rateFunction(fluxes, material):
        if material == vps.Material.Si:
            neutralContrib = -fluxes[0] * params["neutralRate"]
            ionContrib = -fluxes[1] * params["ionRate"]
            return neutralContrib + ionContrib
        elif material == vps.Material.SiO2:
            return -fluxes[1] * 0.05
        return 0.0

    model.setRateFunction(rateFunction)

    # Run process
    process = vps.Process()
    process.setDomain(domain)
    process.setProcessModel(model)
    process.setProcessDuration(1.0)  # 1 second
    process.setFluxEngineType(vps.FluxEngineType.CPU_DISK)
    process.apply()

    # Planarize at y=50 to clean up top surface before comparing
    vps.Planarize(domain, 50.0).apply()

    # Return resulting level set (etched substrate is the last level set)
    return domain.getLevelSets()[-1]

## Step 5: Configure Optimization

Set up the optimization with parameters and distance metric:

In [ ]:
# Create optimization
opt = fit.Optimization(project)
opt.setProcessSequence(etchingProcess)

# Define parameter names
opt.setParameterNames([
    "ionSourcePower",
    "neutralRate",
    "ionEnergy",
    "neutralStickP",
    "ionRate"
])

# Set parameter bounds (based on physical constraints and literature)
opt.setVariableParameters({
    "ionSourcePower": (10.0, 200.0),   # source power (controls angular spread)
    "neutralRate":    (0.001, 20.0),   # neutral etch rate (nm/s)
    "ionEnergy":      (20.0, 200.0),   # eV
    "neutralStickP":  (0.01, 0.9),     # probability
    "ionRate":        (0.01, 20.0)     # ion etch rate (nm/s)
})

# Choose distance metric — CCH (Chamfer) is good for shape matching
opt.setDistanceMetrics(
    primaryMetric="CCH",
    additionalMetrics=["CA"]  # Also track area for reference
)

# Name and document this run
opt.setName("run1_initialCalibration")
opt.setNotes(
    "Initial calibration of SF6/C4F8 etching parameters. "
    "Using CCH metric to match trench profile shape. "
    "100 evaluations with dlib optimizer."
)

print("Optimization configured")

## Step 6: Run Optimization

Execute the optimization. **Runtime: 15–60 minutes** depending on simulation complexity.

In [ ]:
print("\n" + "="*60)
print("STARTING OPTIMIZATION")
print("="*60)

opt.apply(
    numEvaluations=100,      # Try 100 parameter combinations
    saveComparison=True      # Save intermediate results
)

print("\n" + "="*60)
print("OPTIMIZATION COMPLETE")
print("="*60)

## Step 7: Analyze Results

### Load and Display Results

In [ ]:
# Results location
results_dir = os.path.join(
    project.projectPath,
    "optimizationRuns",
    "run1_initialCalibration"
)

# Load best parameters
with open(os.path.join(results_dir, "run1_initialCalibration-final-results.json")) as f:
    results = json.load(f)

print("\n" + "="*60)
print("OPTIMIZATION RESULTS")
print("="*60)
print(f"\nBest Objective Value: {results['bestScore']:.4f}")
print(f"Achieved at Evaluation: {results['bestEvaluation#']}")
print("\nBest Parameters:")
for param, value in results['bestParameters'].items():
    print(f"  {param:20s}: {value:.4f}")

if 'fixedParameters' in results:
    print("\nFixed Parameters:")
    for param, value in results['fixedParameters'].items():
        print(f"  {param:20s}: {value:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Load progress data
progress = pd.read_csv(os.path.join(results_dir, "progressBest.csv"))

# Plot objective value over time
plt.figure(figsize=(10, 6))
plt.plot(progress['evaluationNumber'], progress['objectiveValue'], 'b-', linewidth=2)
plt.xlabel('Evaluation Number')
plt.ylabel('Objective Value (CCH Distance)')
plt.title('Optimization Convergence')
plt.grid(True, alpha=0.3)
plt.savefig(os.path.join(results_dir, "convergence_analysis.png"), dpi=150)
plt.show()
print(f"\nConvergence plot saved to: {results_dir}/convergence_analysis.png")

## Step 8: Validate Results

### Re-run with Best Parameters

Verify the best parameters actually produce good results:

In [ ]:
# Get best parameters
best = results['bestParameters']

# Re-run simulation
validation_domain = vps.Domain(initialDomain)  # Copy initial domain
result_ls = etchingProcess(validation_domain, best)

# Save result
validation_mesh = vls.Mesh()
vls.ToSurfaceMesh(result_ls, validation_mesh).apply()
writer = vls.VTKWriter(validation_mesh)
writer.setFileName(os.path.join(results_dir, "validation_best_result.vtp"))
writer.apply()

print(f"\nValidation result saved to: {results_dir}/validation_best_result.vtp")
print("Open in ParaView to compare with target")

## Key Takeaways

- **Projects organize everything** — All results in one place
- **Process sequences are flexible** — Can be as complex as needed
- **Choose appropriate metrics** — CCH good for shape, CA for area
- **Monitor convergence** — Check plots to see if converged
- **Validate results** — Re-run with best parameters to verify

## Next Steps

- [Tutorial 2: Custom Parameter Evaluation](../docs/tutorials/tutorial-2-custom-evaluation.md) — Explore parameter space
- [Tutorial 3: Sensitivity Analysis](../docs/tutorials/tutorial-3-sensitivity-analysis.md) — Find important parameters
- [Tutorial 4: Multi-Domain Optimization](../docs/tutorials/tutorial-4-multi-domain.md) — Universal parameters